# Week 4 Coding Practice: Can We Predict Penguin Body Mass?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/main/module04/week4_regression_practice.ipynb)

**Estimated time:** 40-50 minutes  
**Format:** ungraded guided practice

In this activity, you will build and evaluate a regression workflow for a practical question:

> Given measurements and basic descriptive information for a penguin, how accurately can we predict its body mass?

You will begin with one predictor, then add numeric and categorical predictors. The goal is not to memorize scikit-learn syntax. The goal is to decide whether the workflow and the claims made from it are trustworthy.

## Learning objectives

By the end of the activity, you should be able to:

1. identify a regression response and plausible predictors from a prediction question;
2. explore a relationship before fitting a model;
3. fit and interpret a simple linear regression;
4. generate predictions and interpret residuals;
5. extend a workflow to multiple numeric and categorical predictors;
6. split raw data before fitting preprocessing steps;
7. compare training and test MAE, RMSE, and R-squared; and
8. communicate predictive evidence without making unsupported causal claims.

### AI-use checkpoint

An AI assistant can generate the code in this notebook quickly. You remain responsible for checking that the response, predictors, split, preprocessing, metrics, and written claims match the question. If course policy permits AI assistance, record what you used it for and what you verified yourself.

## 1. Load and validate the data

We use the course copy of the Palmer Penguins dataset. It contains observations collected near Palmer Station, Antarctica. The notebook first looks for the repository-relative file and otherwise uses the course repository's raw GitHub copy so the same code works in Colab. Missing entries written as `NA` are read as missing values; no source rows are silently removed or changed.

**Source:** Horst AM, Hill AP, Gorman KB (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. Data originally collected by Kristen Gorman and the Palmer Station Long Term Ecological Research program. [Dataset documentation](https://allisonhorst.github.io/palmerpenguins/).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")

local_data = Path("../data/penguins.csv")
remote_data = (
    "https://raw.githubusercontent.com/obscrivn/"
    "DataScience-book/main/data/penguins.csv"
)
data_source = local_data if local_data.exists() else remote_data
penguins = pd.read_csv(data_source, na_values=["NA"])

expected_columns = {
    "species", "island", "bill_length_mm", "bill_depth_mm",
    "flipper_length_mm", "body_mass_g", "sex", "year"
}
assert penguins.shape == (344, 8), "Unexpected dataset shape."
assert set(penguins.columns) == expected_columns, "Unexpected dataset columns."

print(f"Loaded {penguins.shape[0]} rows from {data_source}")
penguins.head()

### Variables used in this activity

| Variable | Role | Meaning |
| --- | --- | --- |
| `body_mass_g` | response | body mass in grams |
| `flipper_length_mm` | numeric predictor | flipper length in millimeters |
| `bill_length_mm` | numeric predictor | bill length in millimeters |
| `bill_depth_mm` | numeric predictor | bill depth in millimeters |
| `species` | categorical predictor | Adelie, Chinstrap, or Gentoo |
| `sex` | categorical predictor | recorded sex |

The response is numeric, so linear regression is a reasonable model to investigate. That does not guarantee that a linear model will predict well.

In [ ]:
model_columns = [
    "body_mass_g", "flipper_length_mm", "bill_length_mm",
    "bill_depth_mm", "species", "sex"
]

print("Missing values in candidate model columns:")
display(penguins[model_columns].isna().sum().to_frame("missing"))
display(penguins[model_columns].describe(include="all").T)

### Reasoning checkpoint: define the model before writing it

Before continuing, write brief answers.

- What is the response variable, including its unit?
- Which predictor would you try first, and why?
- Name one variable that might improve prediction but would *not* prove a cause of body mass.
- Who or what could this dataset fail to represent?

**Your response:**

## 2. Explore before fitting

We will begin with flipper length as one predictor. Predict what the scatterplot will show before running the cell: direction, approximate shape, group structure, and unusual observations.

In [ ]:
explore_data = penguins.dropna(
    subset=["flipper_length_mm", "body_mass_g", "species"]
)
print(f"Scatterplot uses {len(explore_data)} of {len(penguins)} rows.")

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=explore_data,
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",
    style="species",
    palette="colorblind",
    alpha=0.75,
    ax=ax,
)
ax.set(
    title="Body mass generally increases with flipper length",
    xlabel="Flipper length (mm)",
    ylabel="Body mass (g)",
)
ax.legend(title="Species", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Interpret, do not just describe

The plot suggests a positive, roughly linear overall association. It also shows species clusters, so one overall line may hide relevant group structure. The plot does **not** show that lengthening a penguin's flippers would cause its body mass to increase.

**Your check:** What visual evidence supports using a line, and what feature warns that one predictor may be incomplete?

## 3. Fit a simple linear regression

First, we fit a one-predictor model to the complete flipper-length/body-mass pairs. This full-data fit is a worked example for interpreting a line; it is **not** our final estimate of future performance. Held-out evaluation comes later.

In [ ]:
simple_data = penguins.dropna(
    subset=["flipper_length_mm", "body_mass_g"]
).copy()

simple_preview = LinearRegression()
simple_preview.fit(simple_data[["flipper_length_mm"]], simple_data["body_mass_g"])
simple_data["predicted_mass_g"] = simple_preview.predict(
    simple_data[["flipper_length_mm"]]
)
simple_data["residual_g"] = (
    simple_data["body_mass_g"] - simple_data["predicted_mass_g"]
)

print(f"Intercept: {simple_preview.intercept_:,.1f} g")
print(f"Slope: {simple_preview.coef_[0]:,.1f} g per 1 mm of flipper length")
print(f"Mean residual: {simple_data['residual_g'].mean():.6f} g")

### Reasoning checkpoint: interpret the slope

Complete this sentence using the printed slope:

> In this dataset, penguins with flippers 1 mm longer are predicted by the simple model to have body mass about ______ grams higher/lower, on average.

Why is this an association rather than a causal effect? What does the negative intercept mean mathematically, and why should it not be interpreted as a realistic zero-flipper penguin?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.scatterplot(
    data=simple_data, x="flipper_length_mm", y="body_mass_g",
    color="#0072B2", alpha=0.65, ax=axes[0]
)
line_order = simple_data.sort_values("flipper_length_mm")
axes[0].plot(
    line_order["flipper_length_mm"], line_order["predicted_mass_g"],
    color="#D55E00", linewidth=2, label="Fitted line"
)
axes[0].set(
    title="Simple regression fit", xlabel="Flipper length (mm)",
    ylabel="Body mass (g)"
)
axes[0].legend()

sns.scatterplot(
    data=simple_data, x="predicted_mass_g", y="residual_g",
    color="#009E73", alpha=0.65, ax=axes[1]
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Residuals from the simple model",
    xlabel="Predicted body mass (g)", ylabel="Residual: observed - predicted (g)"
)
plt.tight_layout()
plt.show()

### Reasoning checkpoint: read a residual

A positive residual means the observed penguin is heavier than the model predicted; a negative residual means it is lighter.

- Does the residual plot look like completely patternless noise?
- How could the species clusters in the first plot help explain visible structure?
- Why is a mean residual near zero not proof that individual predictions are accurate?

## 4. Split before learned preprocessing

To estimate performance on unseen penguins, we first split the raw predictor rows and response. We stratify by species so each split retains a similar species mix. The random seed makes this instructional split reproducible.

We do **not** fill missing values or encode categories before the split. Medians, most-frequent categories, and category encodings are learned by a pipeline fitted on the training data only.

In [ ]:
predictor_columns = [
    "flipper_length_mm", "bill_length_mm", "bill_depth_mm",
    "species", "sex"
]
model_data = penguins.dropna(subset=["body_mass_g"]).copy()
X = model_data[predictor_columns]
y = model_data["body_mass_g"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=X["species"]
)

assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(model_data)
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
display(
    pd.concat(
        [X_train["species"].value_counts(normalize=True).rename("train"),
         X_test["species"].value_counts(normalize=True).rename("test")],
        axis=1,
    ).round(3)
)

### Critique an AI-suggested workflow

Suppose an assistant suggests this sequence:

```python
X_filled = X.fillna(X.median(numeric_only=True))
X_encoded = pd.get_dummies(X_filled)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y)
```

The code looks plausible, but the missing-value medians and available categories are learned from the full dataset before the test rows are separated. That lets test information influence model preparation. Explain why the test score could then be too optimistic, even though `body_mass_g` was not used to fill the predictors.

## 5. Compare a simple model with a multiple-predictor model

The simple pipeline uses only flipper length. The multiple pipeline adds two bill measurements plus species and sex. Numeric missing values are filled with training medians. Categorical missing values are filled with the training mode, then categories are one-hot encoded.

Dropping the first encoded category creates a reference group. A categorical coefficient therefore represents a comparison with that reference category, holding the other model predictors fixed.

In [ ]:
simple_features = ["flipper_length_mm"]
numeric_features = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
categorical_features = ["species", "sex"]

simple_preprocess = ColumnTransformer(
    [("numeric", SimpleImputer(strategy="median"), simple_features)],
    verbose_feature_names_out=False,
)
simple_model = Pipeline(
    [("preprocess", simple_preprocess), ("model", LinearRegression())]
)

categorical_preprocess = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ]
)
multiple_preprocess = ColumnTransformer(
    [
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", categorical_preprocess, categorical_features),
    ],
    verbose_feature_names_out=False,
)
multiple_model = Pipeline(
    [("preprocess", multiple_preprocess), ("model", LinearRegression())]
)

simple_model.fit(X_train, y_train)
multiple_model.fit(X_train, y_train)
print("Both pipelines were fit using training rows only.")

In [ ]:
feature_names = multiple_model.named_steps["preprocess"].get_feature_names_out()
coefficient_table = (
    pd.DataFrame(
        {
            "model_feature": feature_names,
            "coefficient_g": multiple_model.named_steps["model"].coef_,
        }
    )
    .assign(abs_coefficient_g=lambda frame: frame["coefficient_g"].abs())
    .sort_values("abs_coefficient_g", ascending=False)
)
coefficient_table.drop(columns="abs_coefficient_g").round(1)

### Reasoning checkpoint: coefficients in context

Choose one numeric coefficient and one categorical coefficient. Interpret each in grams while holding the other predictors fixed. For a categorical feature, identify the omitted reference category before interpreting the comparison.

Then answer: why would it be misleading to rank 'importance' by coefficient magnitude when numeric predictors use different units and categorical coefficients represent group comparisons?

## 6. Evaluate training and test performance

We report three complementary measures:

- **MAE** is the average absolute error in grams. It treats each gram of error proportionally.
- **RMSE** is also in grams but gives extra weight to larger misses.
- **R-squared** is the proportion of response variation accounted for by the model relative to predicting the mean. Higher is better, but it does not tell us the typical error in grams.

Test metrics matter most here because the test rows were not used to fit either pipeline.

In [ ]:
def regression_metrics(model, X_values, y_values):
    predictions = model.predict(X_values)
    return {
        "MAE_g": mean_absolute_error(y_values, predictions),
        "RMSE_g": np.sqrt(mean_squared_error(y_values, predictions)),
        "R_squared": r2_score(y_values, predictions),
    }

rows = []
for model_name, model in {
    "Simple: flipper length": simple_model,
    "Multiple: measurements + categories": multiple_model,
}.items():
    for split_name, X_values, y_values in [
        ("train", X_train, y_train),
        ("test", X_test, y_test),
    ]:
        rows.append(
            {"model": model_name, "split": split_name,
             **regression_metrics(model, X_values, y_values)}
        )

performance = pd.DataFrame(rows)
assert np.isfinite(performance[["MAE_g", "RMSE_g", "R_squared"]]).all().all()
performance.round({"MAE_g": 1, "RMSE_g": 1, "R_squared": 3})

### Reasoning checkpoint: is the model useful?

Use the table rather than a general impression.

1. Which model has the lower test MAE and RMSE? By roughly how many grams?
2. Does the multiple model's improvement seem meaningful relative to a typical penguin body mass?
3. Compare each model's training and test metrics. Is there clear evidence of overfitting, or are the differences small enough to be sampling variation?
4. What error would be acceptable depends on use. Name one low-stakes use where this performance might be adequate and one use where more evidence would be needed.

A lower test error supports better prediction for data similar to this test set. It does not prove that the added predictors cause body mass.

In [ ]:
test_predictions = multiple_model.predict(X_test)
test_results = pd.DataFrame(
    {"observed_mass_g": y_test, "predicted_mass_g": test_predictions}
)
test_results["residual_g"] = (
    test_results["observed_mass_g"] - test_results["predicted_mass_g"]
)

low = min(test_results["observed_mass_g"].min(), test_results["predicted_mass_g"].min())
high = max(test_results["observed_mass_g"].max(), test_results["predicted_mass_g"].max())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(
    test_results["observed_mass_g"], test_results["predicted_mass_g"],
    color="#0072B2", alpha=0.75
)
axes[0].plot([low, high], [low, high], linestyle="--", color="black")
axes[0].set(
    title="Multiple model predictions on unseen test rows",
    xlabel="Observed body mass (g)", ylabel="Predicted body mass (g)"
)
axes[1].scatter(
    test_results["predicted_mass_g"], test_results["residual_g"],
    color="#009E73", alpha=0.75
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Test residuals", xlabel="Predicted body mass (g)",
    ylabel="Residual: observed - predicted (g)"
)
plt.tight_layout()
plt.show()

test_results.reindex(test_results["residual_g"].abs().sort_values(ascending=False).index).head()

## 7. Generate new predictions carefully

The pipeline accepts raw values, then applies the training-fitted preprocessing before prediction. These example rows are hypothetical but remain near the ranges represented in the data. Predictions far outside the training range would be extrapolations and deserve much less trust.

In [ ]:
new_penguins = pd.DataFrame(
    [
        {"flipper_length_mm": 190, "bill_length_mm": 39,
         "bill_depth_mm": 18, "species": "Adelie", "sex": "female"},
        {"flipper_length_mm": 220, "bill_length_mm": 49,
         "bill_depth_mm": 15, "species": "Gentoo", "sex": "male"},
    ]
)
new_penguins.assign(
    predicted_body_mass_g=multiple_model.predict(new_penguins).round(0)
)

### Reasoning checkpoint: communicate a prediction

Choose one row and write a two-sentence report: one sentence stating the predicted body mass and one sentence giving a limitation. Do not call the prediction exact, causal, or guaranteed.

Then consider this claim: *The model shows that changing a penguin from one species to another would change its body mass by the corresponding coefficient.* Explain precisely why that causal interpretation is unsupported.

## 8. Independent transfer: test one modeling decision

Choose **one** small change: add `island`, add `year`, remove `sex`, or use a different random seed. Before coding, predict how it might affect test error and interpretability. Then adapt the pipeline, compare the new test metrics with `performance`, and write three short statements:

1. the change you made and why;
2. the evidence from unseen test data; and
3. one limitation that remains.

Keep this cell self-contained. Later validation does not depend on anything you create here.

In [ ]:
# Your independent model comparison goes here.
# Tip: copy the relevant pipeline structure, change one decision, and evaluate on a new split.


## 9. Final workflow and claim audit

Before accepting your own or an AI-generated regression analysis, verify:

- [ ] The response and predictors match the practical question.
- [ ] Exploration happened before modeling, and missing values were reported.
- [ ] Raw rows were split before learned preprocessing.
- [ ] Imputation and categorical encoding were fit on training data only.
- [ ] Training and test metrics were both checked.
- [ ] MAE and RMSE were interpreted in the response unit.
- [ ] A test R-squared was not treated as proof of causation or usefulness for every decision.
- [ ] Predictions stayed near the represented data range, or extrapolation was disclosed.
- [ ] The conclusion states what the model does **and does not** support.

### Exit reflection

Which check in this list is easiest for an AI-generated analysis to hide or skip? What concrete evidence would you request before trusting that analysis?